In [ ]:
import sys
from pathlib import Path

# Allow imports from repository root (run from repo root or notebooks/)
_repo = Path.cwd()
if not (_repo / "run_simulation.py").exists():
    _repo = _repo.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import lognorm

df = pd.read_csv("disaster_to_application_louisville.csv")
data = df["duration_days"]

shape, loc, scale = lognorm.fit(data, floc=0)

x = np.linspace(min(data), max(data), 1000)

plt.hist(data, bins=30, density=True)
plt.plot(x, lognorm.pdf(x, shape, loc, scale))

plt.xlabel("Duration (days)")
plt.ylabel("Density")
plt.title("Lognormal Fit of Marshall Fire Rebuild Curve")
plt.show()

In [ ]:
print(shape)
print(scale)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import lognorm

df = pd.read_csv("Paradise.csv")
data = df["PermitApplicationTime"]

shape, loc, scale = lognorm.fit(data, floc=0)

x = np.linspace(min(data), max(data), 1000)

plt.hist(data, bins=30, density=True)
plt.plot(x, lognorm.pdf(x, shape, loc, scale))

plt.xlabel("Duration (days)")
plt.ylabel("Density")
plt.title("Lognormal Fit of Camp Fire (Paradise) Rebuild Curve")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import lognorm

df = pd.read_csv("SantaRosa.csv")
data = df["PermitApplicationTime"]

shape, loc, scale = lognorm.fit(data, floc=0)

x = np.linspace(min(data), max(data), 1000)

plt.hist(data, bins=30, density=True)
plt.plot(x, lognorm.pdf(x, shape, loc, scale))

plt.xlabel("Duration (days)")
plt.ylabel("Density")
plt.title("Lognormal Fit of Tubbs Fire (Santa Rosa) Rebuild Curve")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import lognorm

df = pd.read_csv("Sonoma.csv")
data = df["PermitApplicationTime"]

shape, loc, scale = lognorm.fit(data, floc=0)

x = np.linspace(min(data), max(data), 1000)

plt.hist(data, bins=30, density=True)
plt.plot(x, lognorm.pdf(x, shape, loc, scale))

plt.xlabel("Duration (days)")
plt.ylabel("Density")
plt.title("Lognormal Fit of Tubbs Fire (Sonoma) Rebuild Curve")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm

x = np.linspace(0, 1500, 500)

cdf = lognorm.cdf(x, shape, loc, scale)

plt.plot(x, cdf)
plt.xlabel("Days since disaster")
plt.ylabel("Fraction of applications submitted")
plt.title("Marshall Fire Rebuild Curve")
plt.show()

In [ ]:
N = 6571   # example total damaged homes

predicted_apps = N * cdf

pdf = lognorm.pdf(x, shape, loc, scale)
daily_submissions = N * pdf

In [ ]:
plt.plot(x, daily_submissions)
plt.xlabel("Days since disaster")
plt.ylabel("Applications per day")
plt.title("Predicted Permit Submission Rate")
plt.show()

In [ ]:
print(shape)
print(loc)
print(scale)

In [ ]:
# Forecast LA lognormal CDF and applications over full recovery

# Use the LA lognormal fit you already estimated:
# shape_la, loc_la, scale_la

df_la = pd.read_csv("disaster_to_application_la.csv")
data_la = df_la["duration_days"].values
shape_la, loc_la, scale_la = lognorm.fit(data_la, floc=0)

N_la = 6571  # total expected applications

# Choose a horizon long enough to see the curve saturate
x_forecast = np.linspace(0, 3000, 1000)  # days since disaster

# Full (unscaled) LA lognormal CDF and PDF
cdf_la_full = lognorm.cdf(x_forecast, shape_la, loc_la, scale_la)
pdf_la_full = lognorm.pdf(x_forecast, shape_la, loc_la, scale_la)

# Forecast cumulative and daily applications
cum_apps_la = N_la * cdf_la_full          # cumulative forecast
daily_apps_la = N_la * pdf_la_full        # daily forecast

plt.figure(figsize=(8, 5))
plt.plot(x_forecast, cdf_la_full, label="LA lognormal CDF (full forecast)")
plt.xlabel("Days since disaster")
plt.ylabel("Fraction of applications submitted")
plt.title("Forecasted LA Lognormal CDF Over Full Recovery")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(x_forecast, cum_apps_la, label="Cumulative applications (forecast)")
plt.xlabel("Days since disaster")
plt.ylabel("Cumulative applications")
plt.title("Forecasted Cumulative LA Applications (N = 6,571)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_la = pd.read_csv("disaster_to_application_la.csv")
data_la = df_la["duration_days"].values
data_la_sorted = np.sort(data_la)
empirical_cdf_la = np.arange(1, len(data_la_sorted) + 1) / len(data_la_sorted)

plt.figure(figsize=(8, 5))
plt.plot(data_la_sorted, empirical_cdf_la, label="Cumulative applications (forecast)")
plt.xlabel("Days since disaster")
plt.ylabel("Cumulative applications")
plt.title("Forecasted Cumulative LA Applications (N = 6,571)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm
from scipy.optimize import curve_fit

# --------------------------------
# Data
# --------------------------------
x = data_la_sorted
y = empirical_cdf_la

# --------------------------------
# Lognormal CDF function
# --------------------------------
def lognormal_cdf(x, sigma, scale):
    return lognorm.cdf(x, sigma, loc=0, scale=scale)

# --------------------------------
# Fit parameters to empirical CDF
# --------------------------------
p0 = [1.0, np.median(x)]  # initial guess

params, cov = curve_fit(
    lognormal_cdf,
    x,
    y,
    p0=p0,
    bounds=(0, np.inf)
)

sigma_hat, scale_hat = params

print("Estimated parameters")
print("sigma =", sigma_hat)
print("scale =", scale_hat)

# --------------------------------
# Forecast horizon
# --------------------------------
x_forecast = np.linspace(0, 3000, 1000)

cdf_forecast = lognorm.cdf(x_forecast, sigma_hat, loc=0, scale=scale_hat)

# --------------------------------
# Plot
# --------------------------------
plt.figure(figsize=(8,5))

plt.step(x, y, where="post", label="Observed cumulative fraction")

plt.plot(x_forecast, cdf_forecast,
         linewidth=2,
         label="Lognormal CDF forecast")

plt.axvline(x.max(), linestyle=":", color="gray",
            label="End of observed data")

plt.xlabel("Days since disaster")
plt.ylabel("Fraction of total applications")
plt.title("Lognormal CDF Fit and Forecast")

plt.xlim(0,3000)
plt.ylim(0,1.05)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm

# Horizon for comparison
x_forecast = np.linspace(0, 3000, 1000)

# Marshall fit (explicit, to avoid overwritten generic variables)
df_marshall = pd.read_csv("disaster_to_application_louisville.csv")
data_marshall = df_marshall["duration_days"].values
shape_m, loc_m, scale_m = lognorm.fit(data_marshall, floc=0)

# Marshall lognormal CDF
cdf_marshall = lognorm.cdf(x_forecast, shape_m, loc_m, scale_m)

# LA lognormal CDF forecast (LA fit: shape_la, loc_la, scale_la)
cdf_la_forecast = lognorm.cdf(x_forecast, shape_la, loc_la, scale_la)

plt.figure(figsize=(8, 5))

plt.plot(x_forecast, cdf_marshall,
         label="Marshall lognormal CDF", linewidth=2)

plt.plot(x_forecast, cdf_la_forecast,
         label="LA lognormal CDF forecast", linestyle="--", linewidth=2)

plt.xlabel("Days since disaster")
plt.ylabel("Fraction of applications submitted")
plt.title("Marshall vs LA Lognormal CDF Forecasts")
plt.xlim(0, 3000)
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import lognorm, kstest, anderson

# Load Marshall Fire / Louisville data
df_marshal = pd.read_csv("disaster_to_application_louisville.csv")
data_marshal = df_marshal["duration_days"].values

# Fit lognormal (Marshall) – if you already have shape, loc, scale you can skip refitting
shape_m, loc_m, scale_m = lognorm.fit(data_marshal, floc=0)

# 1) Kolmogorov–Smirnov test for fitted lognormal
#    H0: data come from Lognormal(shape_m, loc_m, scale_m)
ks_stat, ks_p = kstest(
    data_marshal,
    'lognorm',
    args=(shape_m, loc_m, scale_m)
)

# 2) Anderson–Darling test for lognormal
ad_result = anderson(np.log(data_marshal), dist='norm')  # log of data should be normal
ad_stat = ad_result.statistic
ad_crit = ad_result.critical_values
ad_sig = ad_result.significance_level

# 3) Log-likelihood, AIC, BIC for the fitted lognormal
n = len(data_marshal)
logpdf_vals = lognorm.logpdf(data_marshal, shape_m, loc=loc_m, scale=scale_m)
loglik = np.sum(logpdf_vals)

k = 2  # shape and scale (loc fixed at 0 via floc=0)
aic = 2 * k - 2 * loglik
bic = k * np.log(n) - 2 * loglik

print("Marshall lognormal fit parameters:")
print(f"  shape = {shape_m:.6f}, loc = {loc_m:.6f}, scale = {scale_m:.6f}\n")

print("Kolmogorov–Smirnov test:")
print(f"  KS statistic = {ks_stat:.4f}, p-value = {ks_p:.4f}\n")

print("Anderson–Darling test on log-data vs Normal (lognormal check):")
print(f"  AD statistic = {ad_stat:.4f}")
for sl, cv in zip(ad_sig, ad_crit):
    print(f"  Critical value {cv:.4f} at significance level {sl:.1f}%")
print()

print("Information criteria for lognormal fit:")
print(f"  Log-likelihood = {loglik:.2f}")
print(f"  AIC = {aic:.2f}")
print(f"  BIC = {bic:.2f}")

In [ ]:
print(shape)
print(loc)
print(scale)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm

# Assumes you already have:
# Marshall: shape, loc, scale
# LA:       shape_la, loc_la, scale_la

# Common x-range for PDFs
x_pdf = np.linspace(0, 3000, 1000)

# Marshall lognormal PDF
pdf_marshall = lognorm.pdf(x_pdf, shape, loc=loc, scale=scale)

# LA lognormal PDF
pdf_la = lognorm.pdf(x_pdf, shape_la, loc=loc_la, scale=scale_la)

plt.figure(figsize=(9, 5))

plt.plot(x_pdf, pdf_marshall, label="Marshall lognormal PDF", linewidth=2)
plt.plot(x_pdf, pdf_la, label="LA lognormal PDF", linestyle="--", linewidth=2)

plt.xlabel("Days since disaster")
plt.ylabel("Density")
plt.title("Marshall vs LA Lognormal PDFs")
plt.xlim(0, 3000)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import lognorm

# Load LA data
df_la = pd.read_csv("disaster_to_application_la.csv")
data_la = df_la["duration_days"].values

# Fit a lognormal to LA data (if you already have shape_la, loc_la, scale_la, skip this line)
shape_la, loc_la, scale_la = lognorm.fit(data_la, floc=0)

# x-grid for the PDF (force 0–1000 days)
x = np.linspace(0, 1000, 1000)

# LA lognormal PDF
pdf_la = lognorm.pdf(x, shape_la, loc=loc_la, scale=scale_la)

plt.figure(figsize=(9, 5))

# Histogram of LA data, but only show 0–1000 on x-axis
plt.hist(data_la, bins=30, density=True, alpha=0.7, label="LA data (histogram)")

plt.plot(x, pdf_la, 'r-', linewidth=2, label="LA lognormal PDF (fit)")

plt.xlabel("Duration (days)")
plt.ylabel("Density")
plt.title("LA Lognormal PDF Fit vs Data")
plt.xlim(0, 1000)        # <- key line
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import lognorm, kstest, anderson

# Load Marshall Fire / Louisville data
df_marshall = pd.read_csv("disaster_to_application_louisville.csv")
data_marshall = df_marshall["duration_days"].values

# Fit lognormal (Marshall) – if you already have shape, loc, scale you can skip refitting
shape_m, loc_m, scale_m = lognorm.fit(data_marshall, floc=0)

# 1) Kolmogorov–Smirnov test for fitted lognormal
#    H0: data come from Lognormal(shape_m, loc_m, scale_m)
ks_stat, ks_p = kstest(
    data_marshall,
    "lognorm",
    args=(shape_m, loc_m, scale_m)
)

# 2) Anderson–Darling test for lognormal
ad_result = anderson(np.log(data_marshall), dist="norm")  # log of data should be normal
ad_stat = ad_result.statistic
ad_crit = ad_result.critical_values
ad_sig = ad_result.significance_level

# 3) Log-likelihood, AIC, BIC for the fitted lognormal
n = len(data_marshall)
logpdf_vals = lognorm.logpdf(data_marshall, shape_m, loc=loc_m, scale=scale_m)
loglik = np.sum(logpdf_vals)

k = 2  # shape and scale (loc fixed at 0 via floc=0)
aic = 2 * k - 2 * loglik
bic = k * np.log(n) - 2 * loglik

print("Marshall lognormal fit parameters:")
print(f"  shape = {shape_m:.6f}, loc = {loc_m:.6f}, scale = {scale_m:.6f}\n")

print("Kolmogorov-Smirnov test:")
print(f"  KS statistic = {ks_stat:.4f}, p-value = {ks_p:.4f}\n")

print("Anderson–Darling test on log-data vs Normal (lognormal check):")
print(f"  AD statistic = {ad_stat:.4f}")
for sl, cv in zip(ad_sig, ad_crit):
    print(f"  Critical value {cv:.4f} at significance level {sl:.1f}%")
print()

print("Information criteria for lognormal fit:")
print(f"  Log-likelihood = {loglik:.2f}")
print(f"  AIC = {aic:.2f}")
print(f"  BIC = {bic:.2f}")

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import lognorm
from scipy.optimize import minimize

# --- Data and censoring setup ---
N_total = 6571
censor_time = 425  # days

df_la = pd.read_csv("disaster_to_application_la.csv")
data_la = df_la["duration_days"].values

# Treat all observed applications as exact (<= 425 days)
observed = data_la  # if there are any >425, you can clip or handle separately
n_obs = len(observed)

# --- Negative log-likelihood for right-censored lognormal with loc=0 ---
# Parameterization: sigma > 0, mu is log-mean; scipy: shape = sigma, scale = exp(mu)
def neg_loglik(params):
    sigma, mu = params
    if sigma <= 0:
        return np.inf
    scale = np.exp(mu)

    # log f(t_i)
    logpdf = lognorm.logpdf(observed, s=sigma, loc=0, scale=scale)
    # log S(censor_time)
    S_c = 1.0 - lognorm.cdf(censor_time, s=sigma, loc=0, scale=scale)
    if S_c <= 0:
        return np.inf
    loglik = np.sum(logpdf) + (N_total - n_obs) * np.log(S_c)
    return -loglik

# --- Initial values from naive (uncensored) fit ---
sigma0, loc0, scale0 = lognorm.fit(observed, floc=0)
mu0 = np.log(scale0)

res = minimize(
    neg_loglik,
    x0=np.array([sigma0, mu0]),
    method="L-BFGS-B",
    bounds=[(1e-6, None), (None, None)],
)

sigma_c, mu_c = res.x
scale_c = np.exp(mu_c)

print("Censored lognormal fit for LA (loc = 0):")
print(f"  sigma (shape) = {sigma_c:.6f}")
print(f"  mu (log-mean) = {mu_c:.6f}")
print(f"  scale = exp(mu) = {scale_c:.6f}")
print(f"  success = {res.success}, message = {res.message}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm

# Assumes you already have:
# Marshall: shape, loc, scale
# LA:       shape_la, loc_la, scale_la

# Common x-range for PDFs
x_pdf = np.linspace(0, 3000, 1000)

# Marshall lognormal PDF
pdf_marshall = lognorm.pdf(x_pdf, shape, loc=loc, scale=scale)

# LA lognormal PDF
pdf_la = lognorm.pdf(x_pdf, sigma_c, loc=loc_la, scale=scale_c)

plt.figure(figsize=(9, 5))

plt.plot(x_pdf, pdf_marshall, label="Marshall lognormal PDF", linewidth=2)
plt.plot(x_pdf, pdf_la, label="LA lognormal PDF", linestyle="--", linewidth=2)

plt.xlabel("Days since disaster")
plt.ylabel("Density")
plt.title("Marshall vs LA Lognormal PDFs")
plt.xlim(0, 3000)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import lognorm

N_la = 6571  # total expected LA applications

# --- Load LA data ---
df_la = pd.read_csv("disaster_to_application_la.csv")
data_la = df_la["duration_days"].values
data_la_sorted = np.sort(data_la)
n_la = len(data_la_sorted)

# LA empirical CDF as fraction of total expected LA applications
empirical_cdf_la = np.arange(1, n_la + 1) / N_la

# --- Load Marshall data (empirical) ---
df_marshall = pd.read_csv("disaster_to_application_louisville.csv")
data_marshall = df_marshall["duration_days"].values
data_marshall_sorted = np.sort(data_marshall)
n_marshall = len(data_marshall_sorted)

# Marshall empirical CDF (assume dataset includes all Marshall applications)
empirical_cdf_marshall = np.arange(1, n_marshall + 1) / n_marshall

# --- Horizon for model CDFs ---
x_forecast = np.linspace(0, 3000, 1000)

# Marshall fit (explicit, to avoid overwritten generic variables)
shape_m, loc_m, scale_m = lognorm.fit(data_marshall, floc=0)

# Marshall lognormal CDF
cdf_marshall = lognorm.cdf(x_forecast, shape_m, loc_m, scale_m)

# LA lognormal CDF forecast (LA fit: shape_la, loc_la, scale_la)
cdf_la_forecast = lognorm.cdf(x_forecast, sigma_c, loc_la, scale_c)

plt.figure(figsize=(9, 5))

# Marshall empirical CDF (raw Marshall data)
plt.step(
    data_marshall_sorted, empirical_cdf_marshall, where="post",
    label="Marshall empirical CDF", alpha=0.7
)

# LA empirical CDF (raw LA data, scaled by 6,571)
plt.step(
    data_la_sorted, empirical_cdf_la, where="post",
    label="LA empirical CDF (applications / 6,571)", alpha=0.7
)

# Marshall lognormal CDF
plt.plot(
    x_forecast, cdf_marshall,
    label="Marshall lognormal CDF", linestyle="--", linewidth=2
)

# LA lognormal CDF forecast
plt.plot(
    x_forecast, cdf_la_forecast,
    label="LA lognormal CDF forecast", linestyle="--", linewidth=2
)

plt.xlabel("Days since disaster")
plt.ylabel("Fraction of applications submitted")
plt.title("Comparison of Marshall and LA Fire Application Timing CDFs")
plt.xlim(0, 1500)
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(sigma_c)
print(scale_c)

In [ ]:
from scipy.stats import probplot
import matplotlib.pyplot as plt
import numpy as np

log_data = np.log(data)

fig, ax = plt.subplots(figsize=(6, 6))
probplot(log_data, dist="norm", plot=ax)

ax.set_title("Q–Q plot: log(Marshall durations) vs Normal")
ax.set_xlabel("Theoretical Normal quantiles")
ax.set_ylabel("Ordered log durations")

plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import lognorm, norm, gamma, expon, weibull_min

data = pd.read_csv("disaster_to_application_louisville.csv")["duration_days"].values  # Marshall durations
n = len(data)

results = []

def add_result(name, dist, params, k_effective, note=""):
    # params is the full scipy .fit tuple (shape(s), loc, scale)
    logpdf_vals = dist.logpdf(data, *params)
    loglik = np.sum(logpdf_vals)
    k = k_effective              # number of free parameters
    aic = 2 * k - 2 * loglik
    bic = k * np.log(n) - 2 * loglik
    results.append({
        "model": name,
        "params": params,
        "k": k,
        "loglik": loglik,
        "AIC": aic,
        "BIC": bic,
        "note": note,
    })

# 1) Lognormal, loc fixed at 0 (what you're already using)
shape1, loc1, scale1 = lognorm.fit(data, floc=0)
add_result(
    "lognormal_floc0",
    lognorm,
    (shape1, loc1, scale1),
    k_effective=2,  # shape, scale (loc fixed)
    note="lognorm, loc=0"
)

# 2) Lognormal, loc fixed at 0 (same Marshall convention)
shape2, loc2, scale2 = lognorm.fit(data, floc=0)
add_result(
    "lognormal_free_loc",
    lognorm,
    (shape2, loc2, scale2),
    k_effective=2,  # shape, scale (loc fixed)
    note="lognorm, loc=0 (repeat)"
)

# 3) Normal on raw data
mu_n, sigma_n = norm.fit(data)   # returns (loc=mu, scale=sigma)
add_result(
    "normal_raw",
    norm,
    (mu_n, sigma_n),   # (loc, scale) – NO extra 0
    k_effective=2,
    note="Normal on raw durations"
)

# 4) Gamma
a_g, loc_g, scale_g = gamma.fit(data, floc=0)  # or drop floc=0 if you want free loc
add_result(
    "gamma_floc0",
    gamma,
    (a_g, loc_g, scale_g),
    k_effective=2,  # shape, scale (loc fixed at 0)
    note="Gamma, loc=0"
)

# 5) Exponential (special gamma with shape=1)
loc_e, scale_e = expon.fit(data, floc=0)
add_result(
    "exponential_floc0",
    expon,
    (loc_e, scale_e),
    k_effective=1,  # only scale, loc fixed
    note="Exponential, loc=0"
)

# 6) Weibull
c_w, loc_w, scale_w = weibull_min.fit(data, floc=0)
add_result(
    "weibull_floc0",
    weibull_min,
    (c_w, loc_w, scale_w),
    k_effective=2,
    note="Weibull, loc=0"
)

df_aicbic = pd.DataFrame(results).sort_values("AIC").reset_index(drop=True)
display(df_aicbic[["model", "note", "k", "loglik", "AIC", "BIC"]])